# pyroaring smoke test

Cython wrapper over CRoaring C/C++ compressed-bitmap library.

In [ ]:
# Example 1: BitMap construction + basic membership / cardinality.
from pyroaring import BitMap

bm = BitMap([1, 2, 3, 100, 1000, 1_000_000])
print(f"len(bm) = {len(bm)}")
print(f"min={bm.min()} max={bm.max()}")
print(f"100 in bm: {100 in bm}")
print(f"42 in bm: {42 in bm}")

bm.add(42)
bm.remove(1)
assert 42 in bm
assert 1 not in bm
assert len(bm) == 6

In [ ]:
# Example 2: Set-algebra operations (union / intersection / difference / xor).
from pyroaring import BitMap

a = BitMap(range(0, 100))
b = BitMap(range(50, 150))

union = a | b
inter = a & b
diff = a - b
sym = a ^ b

print(f"|a| = {len(a)}, |b| = {len(b)}")
print(f"|a u b| = {len(union)}")
print(f"|a n b| = {len(inter)}")
print(f"|a \\ b| = {len(diff)}")
print(f"|a ^ b| = {len(sym)}")

assert len(union) == 150
assert len(inter) == 50
assert len(diff) == 50
assert len(sym) == 100

In [ ]:
# Example 3: Serialize / deserialize via the portable CRoaring format.
from pyroaring import BitMap

original = BitMap([1, 7, 19, 255, 1024, 65535, 4_000_000])
blob = original.serialize()
print(f"serialized size = {len(blob)} bytes for {len(original)} elements")

restored = BitMap.deserialize(blob)
assert restored == original
assert list(restored) == list(original)
print(f"round-trip OK: {list(restored)}")

In [ ]:
# Example 4: FrozenBitMap is immutable + hashable; usable as a dict key / set member.
from pyroaring import FrozenBitMap

fb1 = FrozenBitMap([2, 4, 6, 8])
fb2 = FrozenBitMap([2, 4, 6, 8])
fb3 = FrozenBitMap([1, 3, 5, 7])

assert hash(fb1) == hash(fb2)
assert fb1 == fb2
assert fb1 != fb3

registry = {fb1: "evens", fb3: "odds"}
print(registry[fb2])  # equal to fb1, so same key

try:
    fb1.add(10)  # type: ignore[attr-defined]
    raise AssertionError("FrozenBitMap should not support .add()")
except AttributeError as e:
    print(f"correctly immutable: {e}")

In [ ]:
# Example 5: Range constructor + ordered access (indexing, rank, jaccard).
from pyroaring import BitMap

bm = BitMap(range(10, 1000, 3))  # 10, 13, 16, ..., 997
print(f"|bm| = {len(bm)}")
print(f"bm[0] = {bm[0]}, bm[1] = {bm[1]}, bm[-1] = {bm[-1]}")
print(f"rank(100) = {bm.rank(100)}")

other = BitMap(range(500, 1500))
j = bm.jaccard_index(other)
print(f"jaccard(bm, other) = {j:.4f}")

assert bm[0] == 10
assert bm[-1] == 997
assert 0.0 <= j <= 1.0
print("OK")